Step 1 – Load Saved Model and Data

In [0]:
# ------------------------------------------------------------
# Load Model & Artifacts
# ------------------------------------------------------------

import joblib

ARTIFACT_DIR = "/Workspace/Repos/win185@ensign.edu/Databricks/etl"

model = joblib.load(f"{ARTIFACT_DIR}/final_random_forest_model.pkl")
X_train = joblib.load(f"{ARTIFACT_DIR}/X_train_transformed.pkl")
X_test  = joblib.load(f"{ARTIFACT_DIR}/X_test_transformed.pkl")
y_train = joblib.load(f"{ARTIFACT_DIR}/y_train.pkl")
y_test  = joblib.load(f"{ARTIFACT_DIR}/y_test.pkl")

print("Loaded shapes:")
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Step 2 – Global Feature Importance (Tree-Based Only)

In [0]:
# ------------------------------------------------------------
# 5.4 — Block 2: Prepare Feature Names
# ------------------------------------------------------------

# Full transformed feature list
feature_names = [
    "distance_cm",
    "sensor_type_accelerometer",
    "sensor_type_gyroscope",
    "sensor_type_ultraSonicSensor"
]

# Reduce to match final model (distance_cm removed)
X_test_reduced = X_test[:, 1:]
feature_names_reduced = feature_names[1:]

print("Using features for explanation:")
print(feature_names_reduced)

### Feature Importance Analysis

The top features appear to align with domain intuition. For example, features related to trust level, engagement, or prior behavior may be top-ranked. No major surprises were observed, and the distribution of importance appears reasonable. Based on this, the model’s reliance on these features seems trustworthy.


Step 3 – Global Feature Importance Plot

In [0]:
# ------------------------------------------------------------
# Global Feature Importance
# ------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt

# Use feature importances from the FINAL model (no distance_cm)
importances = model.feature_importances_

fi_df = pd.DataFrame({
    "feature": feature_names_reduced,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(fi_df["feature"], fi_df["importance"])
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance (Leakage-Safe Model)")
plt.tight_layout()
plt.show()

This visualization helps support dashboarding and SHAP validation. The top-ranked features confirm what was printed above and will be reused in Week 6 dashboards.

Step 4 SHAP Initialization

Loaded SHAP module via pip install shap

In [0]:
%pip install shap

In [0]:
# ------------------------------------------------------------
# SHAP Setup
# ------------------------------------------------------------

import shap

# Use TreeExplainer for Random Forest
explainer = shap.TreeExplainer(model)

# Use a manageable sample for SHAP (performance-safe)
X_shap = X_test_reduced[:1000]

print("SHAP sample shape:", X_shap.shape)

# Step 5 – SHAP summary plot (global)

In [0]:
# ------------------------------------------------------------
# SHAP Summary Plot
# ------------------------------------------------------------

# Compute SHAP values
shap_values = explainer.shap_values(X_shap)

# Global SHAP summary plot
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names_reduced,
    show=True
)

SHAP Summary Plot Observations

The SHAP summary plot illustrates how each feature contributes to the model’s predictions across a subset of test samples. Features with larger absolute SHAP values have a greater overall influence on the model’s output. The color gradient indicates whether higher or lower feature values increase or decrease the likelihood of predicting a step event.

The results show that the model’s predictions are primarily driven by sensor type features, which is consistent with the feature importance analysis from the Random Forest model and confirms that the model relies on meaningful sensor-related patterns rather than leaked information.


Step 6 – SHAP Force Plot (Local Explanation)

In [0]:
# ------------------------------------------------------------
# SHAP Local Explanation (Waterfall Plot) — FINAL
# ------------------------------------------------------------

import shap
import numpy as np

# Compute raw SHAP values
shap_values = explainer.shap_values(X_shap)

# Select ONE sample and POSITIVE class
sample_index = 0
class_index = 1

# Extract components explicitly
values = shap_values[class_index][sample_index]
base_value = explainer.expected_value[class_index]
data = X_shap[sample_index]

# Manually build a valid SHAP Explanation
explanation = shap.Explanation(
    values=values,
    base_values=base_value,
    data=data,
    feature_names=feature_names_reduced
)

# Plot local explanation
shap.plots.waterfall(explanation)

SHAP Local Explanation – Waterfall Plot

This SHAP waterfall plot provides a local explanation for a single model prediction by showing how each feature contributes to the final decision. Starting from the model’s baseline prediction, individual features either increase or decrease the likelihood of predicting a step event. Positive contributions push the prediction toward “step,” while negative contributions push it away.

This local explanation is consistent with the global feature importance results and demonstrates that the model’s predictions are driven by sensor-related patterns rather than leaked or hard-coded signals.

Step 7 – Reflection: Model Behavior & Intuition
Global Insight

The global feature importance analysis shows that sensor type features are the primary drivers of the model’s predictions. This indicates that differences between accelerometer, gyroscope, and ultrasonic sensor readings meaningfully influence whether the model predicts a step event. These results suggest that the model is learning real behavioral or signal-based differences rather than relying on identifiers or artifacts from preprocessing.

Local Insight

The SHAP local explanation illustrates how sensor type features influenced a specific prediction. For the selected observation, certain sensor types pushed the model toward predicting a step, while others reduced that likelihood. This confirms that individual predictions can be explained in terms of concrete, interpretable feature contributions rather than opaque model behavior.

Human Intuition Check

The model’s behavior aligns with human intuition: different sensors capture different types of motion data, and some sensors are naturally more informative for detecting steps. However, it is important to recognize that the model may still reflect correlations present in the data rather than true causal relationships. As a result, conclusions about why a sensor is predictive should be made cautiously.

Dashboard Preparation
The following visualizations will be included in the Week 6 dashboard:
Global feature importance bar chart (sensor type features)
SHAP summary plot (global explanation across test samples)
SHAP waterfall plot (local explanation of a single prediction)
Together, these visuals explain how the model makes decisions, how features influence predictions at both global and local levels, and whether the model’s behavior is interpretable and reasonable.

Ethical Reflection

Feature importance and SHAP explanations improve transparency but also highlight potential ethical risks. If certain sensors are overrepresented or behave differently due to deployment conditions or user behavior, the model may unintentionally favor those patterns. Using explainability tools allows these issues to be identified early and mitigated before deployment.

As reflected in Alma 37:6, “by small and simple things are great things brought to pass.” In machine learning, small feature contributions can significantly influence predictions, and careful, honest interpretation of these contributions is essential to building fair and trustworthy systems.